# 04 — ETL Ministerio + INE

Construye `fact_criminalidad_agregada` (ANUAL, por CCAA/Provincia) combinando varias fuentes con grano distinto. Cada fuente rellena unas FKs y deja otras a NULL:

| Fuente | métrica | id_tipo_delito | id_demografia | territorio |
|--------|---------|----------------|---------------|------------|
| Ministerio ccaa_raw | hechos_conocidos/esclarecidos/detenciones | ✅ 44 tipologías | NULL | Cataluña |
| Ministerio provincias_raw | hechos_conocidos/esclarecidos/detenciones | ✅ 44 tipologías | NULL | Barcelona / Girona / Lleida / Tarragona |
| Portal edad+sexo | detenciones | ✅ | ✅ edad+sexo | Cataluña |
| INE condenados sexo+edad | condenados | NULL | ✅ sexo+edad | Cataluña |
| INE condenados sexo+nacionalidad | condenados | NULL | ✅ sexo+nac | Cataluña |

**Output:** `data/clean/fact_criminalidad_agregada.csv` (territorios 137-141, las 5 unidades catalanas con el MISMO detalle de 44 tipologías)

**Columnas objetivo:** `id, id_tiempo, id_territorio, id_tipo_delito, id_demografia, total, metrica, fuente`

**Reglas aplicadas:**
- Encoding `utf-8-sig` (Ministerio/Portal con BOM), `utf-8` (INE)
- P1 — separador de miles con punto: `thousands='.'`
- P2 — valores censurados 999/998 → NA (solo Portal e INE)
- P8 — `Edad` del INE con espacio final → `.str.strip()`
- id_tiempo apunta a las filas ANUALES de dim_tiempo (mes=NULL)
- **Se usan los raw uniformemente (ccaa_raw + provincias_raw) → 44 tipologías en los 5 territorios.**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
MIN = RAW / 'ministerio'
INE = RAW / 'ine'

print('CLEAN existe:', CLEAN.exists())

---
## 1. Cargar dimensiones y construir lookups

In [ ]:
dim_tiempo = pd.read_csv(CLEAN / 'dim_tiempo.csv', encoding='utf-8')
dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
dim_tipo_delito = pd.read_csv(CLEAN / 'dim_tipo_delito.csv', encoding='utf-8')
dim_demografia = pd.read_csv(CLEAN / 'dim_demografia.csv', encoding='utf-8')

# --- Tiempo ANUAL: anyo -> id_tiempo (filas con mes nulo) ---
tiempo_anual = dim_tiempo[dim_tiempo['mes'].isna()].set_index('anyo')['id_tiempo'].to_dict()
print('Años anuales disponibles:', sorted(tiempo_anual.keys()))

# --- Territorio canónico Ministerio/INE ---
terr_min = dim_territorio[dim_territorio['fuente'] == 'ministerio_ine']
ID_CATALUNA = int(terr_min[terr_min['provincia'].isna()]['id_territorio'].iloc[0])
ID_BARCELONA = int(terr_min[terr_min['provincia'] == 'Barcelona']['id_territorio'].iloc[0])
# IDs de las otras 3 provincias catalanas (se rellenan desde los *_provincias_raw)
PROV_IDS = {p: int(terr_min[terr_min['provincia'] == p]['id_territorio'].iloc[0]) for p in ['Girona', 'Lleida', 'Tarragona']}
print('ID Cataluña (CCAA):', ID_CATALUNA, '| ID Barcelona:', ID_BARCELONA, '| otras provincias:', PROV_IDS)

# --- Tipo delito Ministerio/INE: descripcio -> id_tipo_delito ---
tipo_min = dim_tipo_delito[dim_tipo_delito['fuente'] == 'ministerio_ine'][['id_tipo_delito', 'descripcio']].copy()
print('Tipos ministerio_ine en dimensión:', len(tipo_min))
assert tipo_min['descripcio'].is_unique, 'descripcio no es única en tipo_min — el merge haría fan-out'

In [ ]:
# --- Demografía: subconjuntos por tipo de match ---
# Portal: (grup_edat, sexo)
demo_portal = dim_demografia[dim_demografia['fuente'] == 'portal_ministerio'][['id_demografia', 'grup_edat', 'sexo']].copy()
# INE edad: (sexo, grup_edat) con nacionalitat nula
demo_ine_edad = dim_demografia[(dim_demografia['fuente'] == 'ine_condenados') & (dim_demografia['nacionalitat'].isna())][['id_demografia', 'sexo', 'grup_edat']].copy()
# INE nacionalidad: (sexo, nacionalitat) con grup_edat nulo
demo_ine_nac = dim_demografia[(dim_demografia['fuente'] == 'ine_condenados') & (dim_demografia['grup_edat'].isna())][['id_demografia', 'sexo', 'nacionalitat']].copy()

print('demo_portal:', len(demo_portal), '| demo_ine_edad:', len(demo_ine_edad), '| demo_ine_nac:', len(demo_ine_nac))
for nombre, d, keys in [('portal', demo_portal, ['grup_edat','sexo']), ('ine_edad', demo_ine_edad, ['sexo','grup_edat']), ('ine_nac', demo_ine_nac, ['sexo','nacionalitat'])]:
    assert not d.duplicated(keys).any(), f'claves duplicadas en demo_{nombre}'
print('Claves de demografía únicas — OK')

---
## 2. Ministerio — hechos conocidos / esclarecidos / detenciones (5 territorios, raw uniforme)

Sin demografía. Tipología penal → id_tipo_delito. Se usan los ficheros `*_ccaa_raw` (Cataluña)
y `*_provincias_raw` (las 4 provincias) → **44 tipologías para los 5 territorios** (consistente).

In [ ]:
# CATALUÑA (CCAA) desde los *_ccaa_raw → 44 tipologías.
# Usar los raw uniformemente evita que Cataluña/Barcelona tengan menos detalle
# que Girona/Lleida/Tarragona.)
ccaa_files = [
    ('ministerio_hechos_conocidos_ccaa_raw.csv',         'hechos_conocidos'),
    ('ministerio_hechos_esclarecidos_ccaa_raw.csv',      'hechos_esclarecidos'),
    ('ministerio_detenciones_investigados_ccaa_raw.csv', 'detenciones'),
]
frames_min = []
misses_tipo = set()
for fname, metrica in ccaa_files:
    df = pd.read_csv(MIN / fname, encoding='utf-8-sig', sep=';', thousands='.')
    df.columns = [c.strip() for c in df.columns]
    df = df[df['Comunidades autónomas'] == 'CATALUÑA'].copy()
    df['total'] = pd.to_numeric(df['Total'], errors='coerce')
    df = df.merge(tipo_min, left_on='Tipología penal', right_on='descripcio', how='left')
    misses_tipo.update(df[df['id_tipo_delito'].isna()]['Tipología penal'].unique())
    out = pd.DataFrame({
        'id_tiempo': df['periodo'].map(tiempo_anual),
        'id_territorio': ID_CATALUNA,
        'id_tipo_delito': df['id_tipo_delito'],
        'id_demografia': pd.NA,
        'total': df['total'],
        'metrica': metrica,
        'fuente': 'ministerio'
    })
    frames_min.append(out)
    pmin, pmax = int(df['periodo'].min()), int(df['periodo'].max())
    n_tip = df['Tipología penal'].nunique()
    print(f'{fname} (Cataluña): {len(out)} filas, periodo {pmin}-{pmax}, tipologías {n_tip}')

print('\nTipologías CCAA SIN match en dim_tipo_delito:', sorted(misses_tipo) if misses_tipo else 'ninguna')

In [ ]:
# Las 4 PROVINCIAS (Barcelona/Girona/Lleida/Tarragona) desde los *_provincias_raw → mismas
# 44 tipologías que Cataluña. Así los 5 territorios son consistentes (mismo detalle).
PROV_IDS_ALL = {'Barcelona': ID_BARCELONA, 'Girona': PROV_IDS['Girona'],
                'Lleida': PROV_IDS['Lleida'], 'Tarragona': PROV_IDS['Tarragona']}
prov_files = [
    ('ministerio_hechos_conocidos_provincias_raw.csv',         'hechos_conocidos'),
    ('ministerio_hechos_esclarecidos_provincias_raw.csv',      'hechos_esclarecidos'),
    ('ministerio_detenciones_investigados_provincias_raw.csv', 'detenciones'),
]
frames_prov = []
misses_tipo_prov = set()
for fname, metrica in prov_files:
    df = pd.read_csv(MIN / fname, encoding='utf-8-sig', sep=';', thousands='.')
    df.columns = [c.strip() for c in df.columns]
    df = df[df['Provincias'].isin(PROV_IDS_ALL.keys())].copy()
    df['total'] = pd.to_numeric(df['Total'], errors='coerce')
    df = df.merge(tipo_min, left_on='Tipología penal', right_on='descripcio', how='left')
    misses_tipo_prov.update(df[df['id_tipo_delito'].isna()]['Tipología penal'].unique())
    out = pd.DataFrame({
        'id_tiempo': df['periodo'].map(tiempo_anual),
        'id_territorio': df['Provincias'].map(PROV_IDS_ALL),
        'id_tipo_delito': df['id_tipo_delito'],
        'id_demografia': pd.NA,
        'total': df['total'],
        'metrica': metrica,
        'fuente': 'ministerio'
    })
    frames_prov.append(out)
    pmin, pmax = int(df['periodo'].min()), int(df['periodo'].max())
    print(f'{fname} (4 provincias): {len(out)} filas, periodo {pmin}-{pmax}')

print('\nTipologías provincias SIN match en dim_tipo_delito:', sorted(misses_tipo_prov) if misses_tipo_prov else 'ninguna')

---
## 3. Portal estadístico — detenciones por tipología, grupo de edad y sexo (Cataluña)

Con demografía. Solo hay nivel CATALUÑA. P2: censurados 999/998 → NA.

In [ ]:
portal = pd.read_csv(
    MIN / 'PORTAL ESTADÍSTICO DE CRIMINALIDAD Detenciones e investigados por tipología penal, grupo de edad y sexo.csv',
    encoding='utf-8-sig', sep=';', thousands='.', low_memory=False
)
portal.columns = [c.strip() for c in portal.columns]
# Filtrar a Cataluña
portal = portal[portal['Comunidades autónomas'] == 'CATALUÑA'].copy()
print('Filas Portal Cataluña:', len(portal))

# total + censura P2
portal['total'] = pd.to_numeric(portal['Total'], errors='coerce').replace([999, 998], np.nan)

# lookup tipo
portal = portal.merge(tipo_min, left_on='Tipología penal', right_on='descripcio', how='left')
miss_tp = portal[portal['id_tipo_delito'].isna()]['Tipología penal'].unique()
print('Tipologías Portal sin match:', sorted(miss_tp) if len(miss_tp) else 'ninguna')

# lookup demografía (grup_edat, sexo)
portal = portal.merge(demo_portal, left_on=['Grupo de edad', 'Sexo'], right_on=['grup_edat', 'sexo'], how='left')
miss_dem = portal[portal['id_demografia'].isna()][['Grupo de edad', 'Sexo']].drop_duplicates()
print('Demografías Portal sin match:', len(miss_dem))

fact_portal = pd.DataFrame({
    'id_tiempo': portal['periodo'].map(tiempo_anual),
    'id_territorio': ID_CATALUNA,
    'id_tipo_delito': portal['id_tipo_delito'],
    'id_demografia': portal['id_demografia'],
    'total': portal['total'],
    'metrica': 'detenciones',
    'fuente': 'portal'
})
print('fact_portal:', len(fact_portal), 'filas')

---
## 4. INE condenados (sexo+edad y sexo+nacionalidad, Cataluña)

Con demografía, sin tipo de delito. Territorio '09 Cataluña'. P8: Edad con espacio final.

In [ ]:
# 4a. INE sexo + edad
ine_edad = pd.read_csv(INE / 'ine_ccaa_condenados_sexo_edad_raw.csv', encoding='utf-8', sep=';', thousands='.')
ine_edad.columns = [c.strip() for c in ine_edad.columns]
ine_edad = ine_edad[ine_edad['Comunidades y ciudades autónomas'] == '09 Cataluña'].copy()
ine_edad['Edad'] = ine_edad['Edad'].str.strip()  # P8
ine_edad['total'] = pd.to_numeric(ine_edad['Total'], errors='coerce').replace([999, 998], np.nan)
ine_edad = ine_edad.merge(demo_ine_edad, left_on=['Sexo', 'Edad'], right_on=['sexo', 'grup_edat'], how='left')
miss_e = ine_edad[ine_edad['id_demografia'].isna()][['Sexo', 'Edad']].drop_duplicates()
print('INE edad — filas:', len(ine_edad), '| demografías sin match:', len(miss_e))
if len(miss_e): print(miss_e.to_string())

fact_ine_edad = pd.DataFrame({
    'id_tiempo': ine_edad['Periodo'].map(tiempo_anual),
    'id_territorio': ID_CATALUNA,
    'id_tipo_delito': pd.NA,
    'id_demografia': ine_edad['id_demografia'],
    'total': ine_edad['total'],
    'metrica': 'condenados',
    'fuente': 'ine'
})
print('fact_ine_edad:', len(fact_ine_edad), 'filas')

In [ ]:
# 4b. INE sexo + nacionalidad
ine_nac = pd.read_csv(INE / 'ine_ccaa_condenados_sexo_nacionalidad_raw.csv', encoding='utf-8', sep=';', thousands='.')
ine_nac.columns = [c.strip() for c in ine_nac.columns]
ine_nac = ine_nac[ine_nac['Comunidades y ciudades autónomas'] == '09 Cataluña'].copy()
ine_nac['total'] = pd.to_numeric(ine_nac['Total'], errors='coerce').replace([999, 998], np.nan)
ine_nac = ine_nac.merge(demo_ine_nac, left_on=['Sexo', 'Nacionalidad'], right_on=['sexo', 'nacionalitat'], how='left')
miss_n = ine_nac[ine_nac['id_demografia'].isna()][['Sexo', 'Nacionalidad']].drop_duplicates()
print('INE nac — filas:', len(ine_nac), '| demografías sin match:', len(miss_n))
if len(miss_n): print(miss_n.to_string())

fact_ine_nac = pd.DataFrame({
    'id_tiempo': ine_nac['Periodo'].map(tiempo_anual),
    'id_territorio': ID_CATALUNA,
    'id_tipo_delito': pd.NA,
    'id_demografia': ine_nac['id_demografia'],
    'total': ine_nac['total'],
    'metrica': 'condenados',
    'fuente': 'ine'
})
print('fact_ine_nac:', len(fact_ine_nac), 'filas')

---
## 5. Unir todo y construir la tabla final

In [ ]:
fact = pd.concat(frames_min + frames_prov + [fact_portal, fact_ine_edad, fact_ine_nac], ignore_index=True)
print('Total filas antes de validar:', len(fact))

# Validar id_tiempo (todos los periodos deben tener fila anual)
n_sin_tiempo = fact['id_tiempo'].isna().sum()
print('Filas sin id_tiempo:', n_sin_tiempo)
assert n_sin_tiempo == 0, 'Hay periodos fuera de 2010-2025'

# total nulo = valor censurado/no disponible → se conserva la fila pero total queda NA
print('Filas con total NA (censurado/no disponible):', fact['total'].isna().sum())

# Tipos: ids a Int64 nullable (id_tipo_delito y id_demografia pueden ser NULL)
fact['id_tiempo'] = fact['id_tiempo'].astype(int)
fact['id_territorio'] = fact['id_territorio'].astype(int)
fact['id_tipo_delito'] = fact['id_tipo_delito'].astype('Int64')
fact['id_demografia'] = fact['id_demografia'].astype('Int64')

fact.insert(0, 'id', range(1, len(fact) + 1))
print('\nShape final:', fact.shape)
print('Territorios usados:', sorted(fact['id_territorio'].unique()))
print(fact.dtypes)
fact.head()

In [ ]:
print('Validaciones:')
print('  Filas por fuente:')
print(fact.groupby(['fuente', 'metrica']).size())
print('\n  FK id_tiempo en rango:', fact['id_tiempo'].between(1, len(dim_tiempo)).all())
print('  FK id_territorio en rango:', fact['id_territorio'].between(1, len(dim_territorio)).all())
td_ok = fact['id_tipo_delito'].dropna().between(1, len(dim_tipo_delito)).all()
dm_ok = fact['id_demografia'].dropna().between(1, len(dim_demografia)).all()
print('  FK id_tipo_delito en rango (no nulos):', td_ok)
print('  FK id_demografia en rango (no nulos):', dm_ok)
print('  Filas duplicadas (sin id):', fact.drop(columns='id').duplicated().sum())

---
## 6. Guardar

In [ ]:
ruta_salida = CLEAN / 'fact_criminalidad_agregada.csv'
fact.to_csv(ruta_salida, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta_salida.name}: {len(fact)} filas')
print('\nListo para continuar con 05_etl_penitenciaria.ipynb')